In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
import pandas as pd

train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

print(train.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


In [3]:
train['Age'] = train['Age'].fillna(train['Age'].median())
test['Age'] = test['Age'].fillna(test['Age'].median())

train['Embarked'] = train['Embarked'].fillna('S')
test['Embarked'] = test['Embarked'].fillna('S')

test['Fare'] = test['Fare'].fillna(test['Fare'].median())

In [4]:
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})

train['Embarked'] = train['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
test['Embarked'] = test['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [5]:
features = ['Pclass', 'Sex', 'Fare', 'Embarked']

X = train[features]
y = train['Survived']
X_test = test[features]

In [6]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X, y)

RandomForestClassifier(max_depth=5, random_state=42)

In [7]:
predictions = model.predict(X_test)

In [8]:
output = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

output.to_csv('submission.csv', index=False)
print("file created")

file created


In [9]:
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
test['FamilySize'] = test['SibSp'] + test['Parch'] +1

In [10]:
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)

In [11]:
import pandas as pd

train['AgeGroup'] = pd.cut(train['Age'], bins=5, labels=False)
test['AgeGroup'] = pd.cut(test['Age'], bins=5, labels=False)

In [12]:
features = ['Pclass', 'Sex', 'Fare', 'Embarked', 
            'FamilySize', 'IsAlone', 'AgeGroup']

In [13]:
X = train[features]
X_test = test[features]
y = train['Survived']

model.fit(X, y)
predictions = model.predict(X_test)

In [14]:
print(train[features].isnull().sum())

Pclass        0
Sex           0
Fare          0
Embarked      0
FamilySize    0
IsAlone       0
AgeGroup      0
dtype: int64


In [15]:
# 4.6 Title Extraction (FIXED)
train['Title'] = train['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)
test['Title']  = test['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)

# Simplify titles
common_titles = ['Mr','Miss','Mrs','Master']

train['Title'] = train['Title'].apply(lambda x: x if x in common_titles else 'Others')
test['Title']  = test['Title'].apply(lambda x: x if x in common_titles else 'Others')

# Convert Title to numbers
title_map = {'Mr':0,'Miss':1,'Mrs':2,'Master':3,'Others':4}

train['Title'] = train['Title'].map(title_map)
test['Title']  = test['Title'].map(title_map)

# =========================
# 5. MODEL
# =========================

features = ['Pclass','Sex','Fare','Embarked','FamilySize','IsAlone','AgeGroup','Title']

X = train[features]
y = train['Survived']
X_test = test[features]

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X, y)

# =========================
# 6. PREDICTION
# =========================

predictions = model.predict(X_test)

# =========================
# 7. SUBMISSION FILE
# =========================

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

submission.to_csv('submission.csv', index=False)

print("✅ DONE - Ready for submission")

✅ DONE - Ready for submission
